#### UHI defined by stations at similar elevations
Even though the stations have been filtered for data availability and the domain has been restricted to only include stations near Montréal, uneven distributions in elevation (or latitude to a lesser extent due to the domain restriction) may still play a factor, so another filtering step is necessary considering the distributions of these properties. 

In [21]:
from Montreal_UHI_toolbox import stations_mtl, dry_stations_mtl, urban_stations_mtl, rural_stations_mtl, suburban_stations_mtl
from scipy.stats import gaussian_kde
from plotly.subplots import make_subplots

import plotly.graph_objects as go
import numpy as np
import pandas as pd

In [3]:
# Filter high/low elevation outliers from mtl stations
mean_elev = np.mean(urban_stations_mtl.elev)
std_elev = np.std(urban_stations_mtl.elev)

# Stations outside of two standard deviation from the mean urban elevation are not used to analyse the UHI
lower_elev = mean_elev - std_elev*2 
upper_elev = mean_elev + std_elev*2
# # Alternatively, and functionally equivalent:
# lower_elev = 0 
# upper_elev = 100

# obs includes all station data used for analysis
obs = dry_stations_mtl.where(dry_stations_mtl.elev >= lower_elev,drop=True).where(dry_stations_mtl.elev <= upper_elev,drop=True)
obs_urban = urban_stations_mtl.where(urban_stations_mtl.elev >= lower_elev,drop=True).where(urban_stations_mtl.elev <= upper_elev,drop=True)
obs_suburban = suburban_stations_mtl.where(suburban_stations_mtl.elev >= lower_elev,drop=True).where(suburban_stations_mtl.elev <= upper_elev,drop=True)
obs_rural = rural_stations_mtl.where(rural_stations_mtl.elev >= lower_elev,drop=True).where(rural_stations_mtl.elev <= upper_elev,drop=True)

In [ ]:
for x_title, indvar, unit in zip(['Elevation','Latitude'],['elev', 'lat'],['m','°N']):
    for ds, station_type in zip(
        [
            [dry_stations_mtl,     obs],
            [urban_stations_mtl,   obs_urban],
            [suburban_stations_mtl,obs_suburban],
            [rural_stations_mtl,   obs_rural]
        ],
        ['All','Urban','Suburban','Rural']
        ):

        before = ds[0].reset_coords('lat', drop=False)[indvar].to_dataframe()
        after  = ds[1].reset_coords('lat', drop=False)[indvar].to_dataframe()

        # Dataframe for overlay
        before_key = f' Before ({len(before)})'
        after_key = f' After ({len(after)})'
        df = pd.concat([
            before.assign(Station=before_key),
            after.assign(Station=after_key)
        ])
        
        # Plot the histogram
        fig = make_subplots(specs=[[{'secondary_y': True}]])
        # Consistent axes and bin sizes
        xmin = 0
        xmax = 400
        binsize=10
        if x_title == 'Latitude':
            xmin = 44
            xmax = 47
            binsize=0.1

        fig.add_trace(go.Histogram(x=before[indvar], xbins=dict(start=xmin,end=xmax,size=binsize),
                            name=before_key, marker_color='grey', opacity=0.3,marker_line_color='black', marker_line_width=1), secondary_y=False)
        fig.add_trace(go.Histogram(x=after[indvar], xbins=dict(start=xmin,end=xmax,size=binsize),
                            name=after_key, marker_color='blue', opacity=0.5,marker_line_color='black', marker_line_width=1), secondary_y=False)
        fig.update_layout(barmode='overlay')
        

        # Displaying KDEs
        pad = 0.1 * (xmax - xmin)
        xk = np.linspace(xmin - pad, xmax + pad,800)

        # KDE before/after elevation outliers are removed
        kde_before = gaussian_kde(before[indvar])
        kde_after = gaussian_kde(after[indvar])

        fig.add_trace(
            go.Scatter(x=xk, y=kde_before(xk),
                mode='lines', line=dict(color='grey', width=2),
                name='KDE (Before)'),
                secondary_y=True)

        fig.add_trace(
            go.Scatter(x=xk, y=kde_after(xk),
                mode='lines', line=dict(color='blue', width=2),
                name='KDE (After)'),
            secondary_y=True)
        
        fig.update_xaxes(range=[xmin,xmax],title=f'{x_title} ({unit})')
        fig.update_yaxes(title='Count',secondary_y=False)
        fig.update_yaxes(title=f'Density (1/{unit})',showgrid=False,secondary_y=True)
        fig.update_layout(title=f'Before and After Elevation Filtering MTL Stations ({station_type})')
        fig.update_yaxes(range=[0, None])
        
        
        # fig.show()
        fig.write_html(f'/home/gulley/UHI_HW_MTL/info/plots/{station_type.lower()}_{indvar.lower()}_distribution_MTL_obs.html')